# Pipeline de Préprocessing des Signaux Vibratoires Industriels

## Contexte et Objectifs

L'objectif de ce notebook est de construire un **dataset propre et exploitable pour la prédiction des gaps / valeurs manquantes dans des signaux vibratoires industriels**.

### Principes et Contraintes :
1. **Pas d'entraînement de modèle** à cette étape.
2. **Conservation de la forme tabulaire** : aucune transformation en fenêtres, séquences, spectrogrammes ou feature engineering.
3. **Traitements appliqués à l'ensemble des colonnes de données** :
   - Ingestion et nettoyage des fichiers CSV (gestion des conflits Git et formats européens).
   - Traitement des `NaN` par **interpolation linéaire des petits gaps** (longueur <= SHORT_GAP_THRESHOLD).
   - **Tronquage des séries** à partir du premier long gap (longueur > SHORT_GAP_THRESHOLD).
   - **Réduction des données `OFF` à 10 %** (pression < 5.0 bar ou état `OFF`) via échantillonnage reproductible.
   - Elimination complète de toutes les valeurs `NaN` résiduelles sur toutes les colonnes (0 NaN restant).
   - Fusion dans un dataset tabulaire unique : `preprocessed_data.csv`.



## 1. Imports & Configuration

Définition des paramètres globaux configurables, notamment le seuil de longueur des gaps (`SHORT_GAP_THRESHOLD`).



In [8]:
import glob
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration des seuils (Facilement modulables)
SHORT_GAP_THRESHOLD = 10           # Seuil max pour interpolation linéaire (petits gaps)
PRESSURE_OPERATIONAL_THRESHOLD = 5.0  # Seuil de pression en bar (séparation OFF / ON)
OFF_KEEP_FRACTION = 0.10            # Conservation de 10% des données OFF
RANDOM_STATE = 42                   # Graine aléatoire fixe pour la reproductibilité

# Chemins des dossiers (compatibles execution depuis notebooks/ ou racine)

DATA_DIR = '../data/raw'
OUTPUT_DIR = '../data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
FINAL_CSV_PATH = os.path.join(OUTPUT_DIR, 'preprocessed_data.csv')
ROOT_CSV_PATH = 'preprocessed_data.csv'

print(f"SHORT_GAP_THRESHOLD: {SHORT_GAP_THRESHOLD}")
print(f"PRESSURE_OPERATIONAL_THRESHOLD: {PRESSURE_OPERATIONAL_THRESHOLD} bar")
print(f"OFF_KEEP_FRACTION: {OFF_KEEP_FRACTION * 100}%")
print(f"DATA_DIR: {DATA_DIR}")



SHORT_GAP_THRESHOLD: 10
PRESSURE_OPERATIONAL_THRESHOLD: 5.0 bar
OFF_KEEP_FRACTION: 10.0%
DATA_DIR: ../data/raw


## 2. Définition des Fonctions Modulaires

Conformément au cahier des charges, le code est organisé en fonctions distinctes et réutilisables. Les opérations sont appliquées à toutes les colonnes de signaux / capteurs.



In [9]:
from pandas.core.interchange import dataframe_protocol
from pandas.core.interchange import dataframe_protocol
from pandas.core.interchange import dataframe_protocol
from pandas.core.interchange import dataframe_protocol
from pandas.core.interchange import dataframe_protocol
from pandas.core.interchange import dataframe_protocol
def parse_european_num(val):
    """
    Parse les valeurs numériques brutes (gestion des virgules européennes, 
    des points de milliers et des chaînes 'OFF' / 'NaN').
    """
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.upper() in ['NAN', 'NONE', '', 'NULL', 'OFF']:
        return np.nan
    s = s.replace(',', '.')
    parts = s.split('.')
    if len(parts) > 2:
        s = ''.join(parts[:-1]) + '.' + parts[-1]
    try:
        return float(s)
    except:
        return np.nan

def load_csv_files(data_dir):
    """
    Parcourt le dossier data_dir, charge les CSV, résout les conflits Git 
    et normalise la structure des colonnes tout en conservant les noms et métadonnées.
    """
    csv_paths = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
    dfs = []
    
    for path in csv_paths:
        with open(path, 'r', encoding='utf-8', errors='ignore') as fp:
            raw = fp.read()
        
        # Traitement des marqueurs de conflits Git
        if '<<<<<<< HEAD' in raw:
            head_end = raw.find('=======')
            raw = raw[len('<<<<<<< HEAD\n'):head_end].strip() + '\n'
            
        first_line = raw.splitlines()[0] if raw.splitlines() else ''
        sep = ';' if ';' in first_line else ','
        
        df_raw = pd.read_csv(io.StringIO(raw), sep=sep, dtype=str)
        filename = os.path.basename(path)
        pump_id = filename[0]  # 'A', 'B', or 'C'
        
        # Standardisation des préfixes de capteurs (A_ACR_Mot.PV -> ACR_Mot.PV)
        new_cols = {}
        for col in df_raw.columns:
            if col.startswith(f'{pump_id}_'):
                new_cols[col] = col[len(f'{pump_id}_'):]
            else:
                new_cols[col] = col
        df_raw = df_raw.rename(columns=new_cols)
        
        # Conversion numérique propre des colonnes de données
        for col in df_raw.columns:
            if col != 'Timestamp':
                df_raw[col] = df_raw[col].apply(parse_european_num)
                
        # Ajout de l'identifiant d'équipement juste après Timestamp
        df_raw.insert(1, 'equipment_id', pump_id)
        dfs.append((filename, df_raw))
        
    return dfs

def detect_gaps(series):
    """
    Détecte toutes les séquences consécutives de NaN (gaps) dans une série.
    Retourne des positions (indices entiers) pour start/end et la longueur.
    """
    arr = series.isna().to_numpy()
    if not arr.any():
        return []

    idxs = np.where(arr)[0]
    gaps = []
    start = idxs[0]
    prev = idxs[0]
    for i in idxs[1:]:
        if i == prev + 1:
            prev = i
            continue
        # clôture du bloc
        gaps.append({'start_pos': start, 'end_pos': prev, 'length': prev - start + 1})
        start = i
        prev = i
    # dernier bloc
    gaps.append({'start_pos': start, 'end_pos': prev, 'length': prev - start + 1})
    return gaps

def classify_gaps(gaps, threshold):
    """
    Classe les gaps en petits gaps (<= threshold) et grands gaps (> threshold).
    Compatibilité avec les clés `start_pos`/`end_pos` et `length`.
    """
    short_gaps = [g for g in gaps if g['length'] <= threshold]
    long_gaps = [g for g in gaps if g['length'] > threshold]
    return short_gaps, long_gaps

def truncate_long_gaps(df, signal_cols, threshold):
    """
    Dès qu'un long gap (> threshold) apparaît dans l'une des colonnes de signaux,
    supprime les données à partir du début de ce long gap jusqu'à la fin de la série.
    Utilise des positions entières pour éviter des comportements imprévus
    lorsqu'un index n'est pas numérique (timestamps).
    """
    earliest_long_gap_pos = None
    long_gaps_found = 0
    short_gaps_found = 0

    n = len(df)
    for col in signal_cols:
        gaps = detect_gaps(df[col])
        short_gaps, long_gaps = classify_gaps(gaps, threshold)
        short_gaps_found += len(short_gaps)
        long_gaps_found += len(long_gaps)

        for lg in long_gaps:
            if earliest_long_gap_pos is None or lg['start_pos'] < earliest_long_gap_pos:
                earliest_long_gap_pos = lg['start_pos']

    if earliest_long_gap_pos is not None:
        rows_removed = n - earliest_long_gap_pos
        df_truncated = df.iloc[:earliest_long_gap_pos].copy()
    else:
        rows_removed = 0
        df_truncated = df.copy()

    return df_truncated, short_gaps_found, long_gaps_found, rows_removed

def interpolate_short_gaps(df, signal_cols, threshold):
    """
    Interpolation linéaire ciblée seulement sur les petits gaps (<= threshold).
    Ne fait PAS de propagation (`ffill` / `bfill`) sur les longs gaps ou aux bords.
    Retourne le DataFrame interpolé et le nombre total de valeurs interpolées.
    """
    df_interp = df.copy()
    interpolated_count = 0
    n = len(df_interp)

    for col in signal_cols:
        series = df_interp[col]
        gaps = detect_gaps(series)
        short_gaps = [g for g in gaps if g['length'] <= threshold]

        for sg in short_gaps:
            start = int(sg['start_pos'])
            end = int(sg['end_pos'])
            length = sg['length']

            left_idx = start - 1
            right_idx = end + 1

            # Si le gap touche un bord ou les valeurs limites sont manquantes, NE PAS interpoler
            if left_idx < 0 or right_idx >= n:
                continue

            left_val = series.iat[left_idx]
            right_val = series.iat[right_idx]
            if pd.isna(left_val) or pd.isna(right_val):
                continue

            interp_values = np.linspace(left_val, right_val, num=length + 2)[1:-1]
            df_interp.iloc[start:end+1, df_interp.columns.get_loc(col)] = interp_values
            interpolated_count += length

    return df_interp, interpolated_count

def reduce_off_data(df, pressure_threshold, keep_frac, random_state):
    """
    Identifie les lignes OFF (pression < pressure_threshold ou indicateur OFF).
    Conserve exactement keep_frac (10%) des données OFF et supprime les 90% restantes.
    """
    off_mask = (df['Pres.PV'] < pressure_threshold) | (df.isin(['OFF']).any(axis=1))
    off_rows_before = off_mask.sum()
    
    df_on = df[~off_mask].copy()
    df_off = df[off_mask].copy()
    
    df_off_kept = df_off.sample(frac=keep_frac, random_state=random_state)
    off_rows_kept = len(df_off_kept)
    off_rows_removed = off_rows_before - off_rows_kept
    
    df_reduced = (
        pd.concat([df_on, df_off_kept], ignore_index=True)
        .sort_values(['equipment_id', 'Timestamp'])
        .reset_index(drop=True)
    )
    
    return df_reduced, off_rows_before, off_rows_kept, off_rows_removed

def merge_datasets(dfs):
    """
    Regroupe les DataFrames traités dans un seul dataset.
    """
    return pd.concat(dfs, ignore_index=True)

def validate_dataset(df_before_total, df_final, signal_cols, threshold):
    """
    Vérifie la qualité du dataset prétraité.
    """
    remaining_nans = df_final[signal_cols].isna().sum().sum()
    
    # Vérification des longs gaps restants dans les signaux
    remaining_long_gaps = 0
    for col in signal_cols:
        gaps = detect_gaps(df_final[col])
        _, long_gaps = classify_gaps(gaps, threshold)
        remaining_long_gaps += len(long_gaps)
        
    print("===== VERIFICATION QUALITE =====")
    print(f"NaNs restants sur toutes les colonnes : {remaining_nans}")
    print(f"Longs gaps non traités restants : {remaining_long_gaps}")
    print(f"Nombre de colonnes conservées : {df_final.shape[1]}")
    print("================================")
    return remaining_nans == 0 and remaining_long_gaps == 0

PHYSICAL_RULES = {
    'ACR_Mot.SV': (0.0, 50.0),
    'ACR_Pmp.SV': (0.0, 50.0),
    'ACR_Mot.PV': (0.0, 10.0),
    'ACR_Pmp.PV': (0.0, 10.0),
    'Pres.PV': (0.0, 100.0),
    'Temp.PV': (-10.0, 200.0),
    'ACR_Mot.TV': (-10.0, 200.0),
    'ACR_Pmp.TV': (-10.0, 200.0),
    'Barometer': (800.0, 1200.0),
    'Temperature': (-10.0, 60.0),
}

def invalid_physical_values(df):
    for sensor, (min_val, max_val) in PHYSICAL_RULES.items():
        if sensor not in df.columns:
            continue

        df[sensor] = pd.to_numeric(
            df[sensor],
            errors="coerce"
        )

        if min_val is not None:
            df.loc[df[sensor] < min_val, sensor] = np.nan

        if max_val is not None:
            df.loc[df[sensor] > max_val, sensor] = np.nan
    return df

def save_dataset(df, path):
    """
    Sauvegarde le DataFrame dans le fichier CSV final.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    # Sauvegarde également à la racine pour accès direct
    df.to_csv(ROOT_CSV_PATH, index=False)
    print(f"Dataset sauvegardé avec succès dans : {path} et {ROOT_CSV_PATH}")

def remove_remaining_nans(df, signal_cols):
    """
    Supprime toutes les lignes contenant NaN dans `signal_cols`.
    Retourne le DataFrame nettoyé et le nombre de lignes supprimées.
    """
    before = len(df)
    df_clean = df.dropna(subset=signal_cols).reset_index(drop=True)
    removed = before - len(df_clean)
    return df_clean, removed

def generate_report(report_dict):
    """
    Affiche le rapport détaillé du preprocessing.
    """
    report_text = f"""
===== PREPROCESSING REPORT =====

Number of input CSV files: {report_dict['input_files']}
Total rows before preprocessing: {report_dict['rows_before']}
Total rows after preprocessing: {report_dict['rows_after']}

Number of short gaps: {report_dict['short_gaps']}
Number of short gaps interpolated: {report_dict['short_gaps_interp']}

Number of long gaps: {report_dict['long_gaps']}
Number of rows removed because of long gaps: {report_dict['rows_removed_long_gaps']}

Number of OFF rows before: {report_dict['off_rows_before']}
Number of OFF rows kept: {report_dict['off_rows_kept']}
Number of OFF rows removed: {report_dict['off_rows_removed']}

Number of remaining NaN values: {report_dict['remaining_nans']}

Final number of rows: {report_dict['rows_after']}
Final number of columns: {report_dict['final_cols']}
"""
    print(report_text)
    return report_text



## 3. Exécution du Pipeline de Prétraitement

Exécution séquentielle du chargement, du filtrage des gaps sur toutes les colonnes, de l'interpolation, de la réduction des données `OFF` et de la consolidation.



In [10]:
# 1. Chargement des fichiers CSV
file_dfs = load_csv_files(DATA_DIR)
num_files = len(file_dfs)
total_rows_before = sum(len(df) for _, df in file_dfs)

# Ensemble des colonnes numériques de données / capteurs
signal_cols = [c for c in file_dfs[0][1].columns if c not in ['Timestamp', 'equipment_id']]

total_short_gaps = 0
total_short_gaps_interp = 0
total_long_gaps = 0
total_rows_removed_long_gaps = 0

processed_dfs = []

# 2. Traitement par fichier (Tronquage longs gaps & Interpolation petits gaps)
for fname, df in file_dfs:
    # Tronquage des séries à partir du premier long gap sur n'importe quel capteur
    df_trunc, s_gaps, l_gaps, r_removed = truncate_long_gaps(df, signal_cols, SHORT_GAP_THRESHOLD)
    total_short_gaps += s_gaps
    total_long_gaps += l_gaps
    total_rows_removed_long_gaps += r_removed

    df_trunc = invalid_physical_values(df_trunc)
    
    # Interpolation linéaire des petits gaps restants dans la partie conservée
    df_interp, s_interp = interpolate_short_gaps(df_trunc, signal_cols, SHORT_GAP_THRESHOLD)
    total_short_gaps_interp += s_interp
    
    processed_dfs.append(df_interp)

# 3. Fusion temporaire
df_merged = merge_datasets(processed_dfs)

# 4. Réduction des données OFF à 10%
df_final, off_before, off_kept, off_removed = reduce_off_data(
    df_merged, 
    PRESSURE_OPERATIONAL_THRESHOLD, 
    OFF_KEEP_FRACTION, 
    RANDOM_STATE
)

# 5. Supprimer les NaN restantes sur toutes les colonnes de signaux
# (ne touche pas à 'Timestamp' et 'equipment_id')
df_final, rows_removed_due_to_nans = remove_remaining_nans(df_final, signal_cols)

# 6. Validation et sauvegarde (Vérification sur toutes les colonnes)
remaining_nans = df_final[signal_cols].isna().sum().sum()
ok = validate_dataset(total_rows_before, df_final, signal_cols, SHORT_GAP_THRESHOLD)
if not ok:
    print("Attention: la validation a échoué — des NaN ou longs gaps subsistent.")

save_dataset(df_final, FINAL_CSV_PATH)

# 7. Report final
report_data = {
    'input_files': num_files,
    'rows_before': total_rows_before,
    'rows_after': len(df_final),
    'short_gaps': total_short_gaps,
    'short_gaps_interp': total_short_gaps_interp,
    'long_gaps': total_long_gaps,
    'rows_removed_long_gaps': total_rows_removed_long_gaps,
    'off_rows_before': off_before,
    'off_rows_kept': off_kept,
    'off_rows_removed': off_removed,
    'rows_removed_due_to_nans': rows_removed_due_to_nans,
    'remaining_nans': remaining_nans,
    'final_cols': df_final.shape[1]
}

generate_report(report_data)



===== VERIFICATION QUALITE =====
NaNs restants sur toutes les colonnes : 0
Longs gaps non traités restants : 0
Nombre de colonnes conservées : 12
Dataset sauvegardé avec succès dans : ../data/processed\preprocessed_data.csv et preprocessed_data.csv

===== PREPROCESSING REPORT =====

Number of input CSV files: 9
Total rows before preprocessing: 173730
Total rows after preprocessing: 15686

Number of short gaps: 163
Number of short gaps interpolated: 78

Number of long gaps: 92
Number of rows removed because of long gaps: 131508

Number of OFF rows before: 28440
Number of OFF rows kept: 2844
Number of OFF rows removed: 25596

Number of remaining NaN values: 0

Final number of rows: 15686
Final number of columns: 12



'\n===== PREPROCESSING REPORT =====\n\nNumber of input CSV files: 9\nTotal rows before preprocessing: 173730\nTotal rows after preprocessing: 15686\n\nNumber of short gaps: 163\nNumber of short gaps interpolated: 78\n\nNumber of long gaps: 92\nNumber of rows removed because of long gaps: 131508\n\nNumber of OFF rows before: 28440\nNumber of OFF rows kept: 2844\nNumber of OFF rows removed: 25596\n\nNumber of remaining NaN values: 0\n\nFinal number of rows: 15686\nFinal number of columns: 12\n'

In [11]:
print("DATA_DIR:", DATA_DIR)

file_dfs = load_csv_files(DATA_DIR)

print("Number of files loaded:", len(file_dfs))

if file_dfs:
    print("First file:", file_dfs[0][0])
    print("Columns:", file_dfs[0][1].columns.tolist())
else:
    print("ERROR: No CSV files were loaded.")

DATA_DIR: ../data/raw
Number of files loaded: 9
First file: A_2024-04-10.csv
Columns: ['Timestamp', 'equipment_id', 'ACR_Mot.PV', 'ACR_Mot.SV', 'ACR_Mot.TV', 'ACR_Pmp.PV', 'ACR_Pmp.SV', 'ACR_Pmp.TV', 'Pres.PV', 'Temp.PV', 'Barometer', 'Temperature']


## 4. Aperçu du Dataset Final Tabulaire

Vérification des premières lignes et des comptages de valeurs nulles de `preprocessed_data.csv`.



In [12]:
print('Aperçu du dataset prétraité final :')
print(f'Forme : {df_final.shape}')
print(f'Comptage global de NaN : {df_final.isna().sum().sum()}')
df_final.head(10)



Aperçu du dataset prétraité final :
Forme : (15686, 12)
Comptage global de NaN : 0


,Timestamp,equipment_id,ACR_Mot.PV,ACR_Mot.SV,ACR_Mot.TV,ACR_Pmp.PV,ACR_Pmp.SV,ACR_Pmp.TV,Pres.PV,Temp.PV,Barometer,Temperature
0,2024-04-10T12:00:30Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.451494,18.061707,1018.053265,20.412823
1,2024-04-10T12:02:50Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.452516,18.102875,1018.078814,20.385832
2,2024-04-10T12:02:55Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.453469,18.102875,1018.078814,20.385832
3,2024-04-10T12:03:35Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.453469,18.215271,1018.078814,20.393339
4,2024-04-10T12:03:50Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.453469,18.215271,1018.028774,20.393339
5,2024-04-10T12:04:20Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.453424,18.254120,1018.028774,20.393339
6,2024-04-10T12:08:00Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.454382,18.120605,1018.032722,20.185040
7,2024-04-10T12:08:15Z,A,0.001184,0.426753,17.976562,0.000464,0.607295,18.218750,0.454382,18.120605,1018.032722,20.185040
8,2024-06-11T10:00:10Z,A,0.001380,0.543140,23.765625,0.000673,0.566808,27.148438,0.828561,23.962830,1012.170715,25.405508
9,2024-06-11T10:00:16Z,A,0.001380,0.543140,23.765625,0.000673,0.566808,27.148438,0.828561,23.962830,1012.176380,25.405731


In [13]:
df_final.describe(include='all')

,Timestamp,equipment_id,ACR_Mot.PV,ACR_Mot.SV,ACR_Mot.TV,ACR_Pmp.PV,ACR_Pmp.SV,ACR_Pmp.TV,Pres.PV,Temp.PV,Barometer,Temperature
count,15686,15686,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000,15686.000000
unique,15633,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2024-06-11T10:05:20Z,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,2,10342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,0.002185,13.710205,33.592024,0.002872,21.629912,33.777091,36.395911,33.472410,1017.903602,23.884231
std,NaN,NaN,0.000980,9.932659,10.214518,0.001310,14.912302,6.425036,13.472976,8.710358,4.266254,1.537564
min,NaN,NaN,0.000533,0.387957,17.718750,0.000255,0.413133,17.609375,0.424722,18.061707,1010.347385,19.007975
25%,NaN,NaN,0.001197,0.506138,24.203125,0.001091,0.566808,30.023438,39.183369,25.818359,1012.060680,23.415518
50%,NaN,NaN,0.002650,15.169131,32.054688,0.003649,30.162298,35.320312,40.302795,33.243469,1020.280371,24.288061
75%,NaN,NaN,0.003018,19.126297,43.328125,0.003838,33.887039,39.117188,42.824001,41.783844,1020.921440,24.775757


In [14]:
df_final.isna().sum()


Timestamp       0
equipment_id    0
ACR_Mot.PV      0
ACR_Mot.SV      0
ACR_Mot.TV      0
ACR_Pmp.PV      0
ACR_Pmp.SV      0
ACR_Pmp.TV      0
Pres.PV         0
Temp.PV         0
Barometer       0
Temperature     0
dtype: int64

In [ ]:
df_final.to_csv("../data/processed/preprocessed_data1.csv", index=False)